In [14]:
import pandas as pd
import numpy as np
import requests
import io, zipfile

In [15]:
def load_month(month):
    url = f"https://data.binance.vision/data/spot/monthly/klines/BTCUSDT/1h/BTCUSDT-1h-{month}.zip"
    r = requests.get(url)
    if r.status_code == 404:
        return None
    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        name = z.namelist()[0]
        m = pd.read_csv(z.open(name), header=None)
    return m

In [16]:
months = pd.period_range("2020-01", "2026-05", freq="M").strftime("%Y-%m")

frames = []
for month in months:
    m = load_month(month)
    if m is not None:
        frames.append(m)
        print(f"loaded {month}: {len(m)} rows")

raw = pd.concat(frames, ignore_index=True)
print("total rows:", len(raw))

loaded 2020-01: 744 rows
loaded 2020-02: 690 rows
loaded 2020-03: 743 rows
loaded 2020-04: 718 rows
loaded 2020-05: 744 rows
loaded 2020-06: 717 rows
loaded 2020-07: 744 rows
loaded 2020-08: 744 rows
loaded 2020-09: 720 rows
loaded 2020-10: 744 rows
loaded 2020-11: 719 rows
loaded 2020-12: 740 rows
loaded 2021-01: 744 rows
loaded 2021-02: 671 rows
loaded 2021-03: 743 rows
loaded 2021-04: 715 rows
loaded 2021-05: 744 rows
loaded 2021-06: 720 rows
loaded 2021-07: 744 rows
loaded 2021-08: 740 rows
loaded 2021-09: 718 rows
loaded 2021-10: 744 rows
loaded 2021-11: 720 rows
loaded 2021-12: 744 rows
loaded 2022-01: 744 rows
loaded 2022-02: 672 rows
loaded 2022-03: 744 rows
loaded 2022-04: 720 rows
loaded 2022-05: 744 rows
loaded 2022-06: 720 rows
loaded 2022-07: 744 rows
loaded 2022-08: 744 rows
loaded 2022-09: 720 rows
loaded 2022-10: 744 rows
loaded 2022-11: 720 rows
loaded 2022-12: 744 rows
loaded 2023-01: 744 rows
loaded 2023-02: 672 rows
loaded 2023-03: 743 rows
loaded 2023-04: 720 rows


In [17]:
raw.columns = ["open_time", "open", "high", "low", "close", "volume",
    "close_time", "quote_volume", "trades",
    "taker_base", "taker_quote", "ignore"]

data = raw[["open_time", "open", "high", "low", "close", "volume"]].copy()

t = data["open_time"].astype("int64")
t_us = np.where(t > 1e14, t, t * 1000)
data["open_time"] = pd.to_datetime(t_us, unit="us", utc=True)
data = data.set_index("open_time").sort_index()
data.index.name = "time"

data["y"] = np.log(data["close"]).shift(-1) - np.log(data["close"])
data = data.iloc[:-1]

data.head()

,open,high,low,close,volume,y
time,,,,,,
2020-01-01 00:00:00+00:00,7195.24,7196.25,7175.46,7177.02,511.814901,0.005454
2020-01-01 01:00:00+00:00,7176.47,7230.00,7175.71,7216.27,883.052603,0.003677
2020-01-01 02:00:00+00:00,7215.52,7244.87,7211.41,7242.85,655.156809,-0.002466
2020-01-01 03:00:00+00:00,7242.66,7245.00,7220.00,7225.01,783.724867,-0.001072
2020-01-01 04:00:00+00:00,7225.00,7230.00,7215.03,7217.27,467.812578,0.000961


In [18]:
data["ret_1"] = np.log(data["close"]) - np.log(data["close"].shift(1))

data["mom_6"]  = data["ret_1"].rolling(6).sum()
data["mom_24"] = data["ret_1"].rolling(24).sum()

data["vol_24"] = data["ret_1"].rolling(24).std()

data[["ret_1", "mom_6", "mom_24", "vol_24", "y"]].tail()

,ret_1,mom_6,mom_24,vol_24,y
time,,,,,
2026-05-31 18:00:00+00:00,-0.001436,-0.004288,-0.005956,0.001385,-0.000547
2026-05-31 19:00:00+00:00,-0.000547,-0.004197,-0.006142,0.001386,0.002393
2026-05-31 20:00:00+00:00,0.002393,0.001866,-0.003252,0.001486,0.002071
2026-05-31 21:00:00+00:00,0.002071,0.003217,-0.001210,0.001553,0.000705
2026-05-31 22:00:00+00:00,0.000705,0.005675,0.000830,0.001535,-0.003661


In [19]:
feature_cols = ["ret_1", "mom_6", "mom_24", "vol_24"]

window = 24 * 30

normed = pd.DataFrame(index=data.index)
for col in feature_cols:
    roll = data[col].rolling(window)
    mean = roll.mean()
    std = roll.std()
    normed[col + "_z"] = (data[col] - mean) / (std + 1e-8)

normed["y"] = data["y"]
normed.tail()

,ret_1_z,mom_6_z,mom_24_z,vol_24_z,y
time,,,,,
2026-05-31 18:00:00+00:00,-0.412830,-0.524791,-0.316619,-1.544849,-0.000547
2026-05-31 19:00:00+00:00,-0.140779,-0.512186,-0.327531,-1.538770,0.002393
2026-05-31 20:00:00+00:00,0.757816,0.332031,-0.120147,-1.445159,0.002071
2026-05-31 21:00:00+00:00,0.660373,0.519170,0.027874,-1.381584,0.000705
2026-05-31 22:00:00+00:00,0.240152,0.860494,0.176234,-1.392938,-0.003661


In [20]:
feature_z = ["ret_1_z", "mom_6_z", "mom_24_z", "vol_24_z"]

dataset = normed[feature_z + ["y"]].dropna()

print(dataset.shape)
print(dataset.index.min(), "to", dataset.index.max())
dataset.head()

(55457, 5)
2020-01-31 23:00:00+00:00 to 2026-05-31 22:00:00+00:00


,ret_1_z,mom_6_z,mom_24_z,vol_24_z,y
time,,,,,
2020-01-31 23:00:00+00:00,-0.992151,0.389432,-0.953006,-0.745971,0.003222
2020-02-01 00:00:00+00:00,0.501884,0.340305,-0.678312,-0.743919,0.004974
2020-02-01 01:00:00+00:00,0.807793,0.820064,-0.511427,-0.671437,0.001274
2020-02-01 02:00:00+00:00,0.156646,0.767896,-0.330720,-0.701558,-0.001012
2020-02-01 03:00:00+00:00,-0.246322,0.344680,-0.367708,-0.700915,-0.003282


In [21]:
shuffled = dataset["y"].sample(frac=1, random_state=0).values
corr = np.corrcoef(dataset["ret_1_z"], dataset["y"])[0, 1]
corr_shuffled = np.corrcoef(dataset["ret_1_z"], shuffled)[0, 1]
print("real corr:", round(corr, 4))
print("shuffled corr:", round(corr_shuffled, 4))

real corr: -0.0165
shuffled corr: -0.0039


In [22]:
def walk_forward_splits(n, train_size, test_size, purge, embargo):
    start = 0
    while start + train_size + purge + embargo + test_size <= n:
        train_end = start + train_size
        test_start = train_end + purge + embargo
        test_end = test_start + test_size

        train_indx = np.arange(start, train_end)
        test_indx = np.arange(test_start, test_end)
        yield train_indx, test_indx

        start += test_size

In [23]:
n = len(dataset)
train_size = 24 * 365
test_size  = 24 * 60
purge      = 24
embargo    = 24

folds = list(walk_forward_splits(n, train_size, test_size, purge, embargo))
print("number of folds:", len(folds))

for i, (tr, te) in enumerate(folds):
    print(
        f"fold {i}: "
        f"train {dataset.index[tr[0]].date()} to {dataset.index[tr[-1]].date()}  |  "
        f"test {dataset.index[te[0]].date()} to {dataset.index[te[-1]].date()}"
    )

number of folds: 32
fold 0: train 2020-01-31 to 2021-01-31  |  test 2021-02-02 to 2021-04-03
fold 1: train 2020-04-01 to 2021-04-01  |  test 2021-04-03 to 2021-06-02
fold 2: train 2020-05-31 to 2021-05-31  |  test 2021-06-02 to 2021-08-01
fold 3: train 2020-07-30 to 2021-07-30  |  test 2021-08-01 to 2021-10-01
fold 4: train 2020-09-28 to 2021-09-29  |  test 2021-10-01 to 2021-11-30
fold 5: train 2020-11-27 to 2021-11-28  |  test 2021-11-30 to 2022-01-29
fold 6: train 2021-01-26 to 2022-01-27  |  test 2022-01-29 to 2022-03-30
fold 7: train 2021-03-27 to 2022-03-28  |  test 2022-03-30 to 2022-05-29
fold 8: train 2021-05-26 to 2022-05-27  |  test 2022-05-29 to 2022-07-28
fold 9: train 2021-07-25 to 2022-07-26  |  test 2022-07-28 to 2022-09-26
fold 10: train 2021-09-24 to 2022-09-24  |  test 2022-09-26 to 2022-11-25
fold 11: train 2021-11-23 to 2022-11-23  |  test 2022-11-25 to 2023-01-24
fold 12: train 2022-01-22 to 2023-01-22  |  test 2023-01-24 to 2023-03-25
fold 13: train 2022-03-23 to

In [24]:
from sklearn.linear_model import LinearRegression

def evaluate_baselines(dataset, folds, feature_cols):
    rows = []
    for i, (tr, te) in enumerate(folds):
        X_tr, y_tr = dataset[feature_cols].values[tr], dataset["y"].values[tr]
        X_te, y_te = dataset[feature_cols].values[te], dataset["y"].values[te]

        pred_zero = np.zeros_like(y_te)
        lin = LinearRegression().fit(X_tr, y_tr)
        pred_lin = lin.predict(X_te)

        acc_lin = np.mean(np.sign(pred_lin) == np.sign(y_te))
        ic_lin  = np.corrcoef(pred_lin, y_te)[0, 1]

        rows.append({"fold": i, "acc_lin": acc_lin, "ic_lin": ic_lin})
    return pd.DataFrame(rows)

feature_cols = ["ret_1_z", "mom_6_z", "mom_24_z", "vol_24_z"]
results = evaluate_baselines(dataset, folds, feature_cols)
print(results.round(4))
print("\nmeans:")
print(results.mean().round(4))

    fold  acc_lin  ic_lin
0      0   0.5062  0.0276
1      1   0.5035  0.0134
2      2   0.5306 -0.0118
3      3   0.5319  0.0119
4      4   0.5271 -0.0169
5      5   0.5188 -0.0065
6      6   0.5083 -0.0097
7      7   0.4868 -0.0401
8      8   0.4812 -0.0272
9      9   0.5056  0.0309
10    10   0.4833 -0.0177
11    11   0.4708  0.0237
12    12   0.4889  0.0660
13    13   0.4750 -0.0656
14    14   0.4806  0.0185
15    15   0.5083  0.0377
16    16   0.5396  0.0634
17    17   0.5062  0.0273
18    18   0.5083 -0.0265
19    19   0.5389  0.0257
20    20   0.5292  0.0275
21    21   0.5111 -0.0133
22    22   0.5319  0.0600
23    23   0.5160  0.0242
24    24   0.5250  0.0103
25    25   0.5389  0.0772
26    26   0.5076  0.0368
27    27   0.5181  0.0246
28    28   0.5062 -0.0320
29    29   0.5111  0.0049
30    30   0.5153 -0.0166
31    31   0.4965  0.0062

means:
fold       15.5000
acc_lin     0.5096
ic_lin      0.0104
dtype: float64


In [25]:
baseline_acc = results["acc_lin"].mean()
baseline_ic  = results["ic_lin"].mean()
print("baseline bar -> acc:", round(baseline_acc, 4), " ic:", round(baseline_ic, 4))

baseline bar -> acc: 0.5096  ic: 0.0104


In [26]:
import torch
import torch.nn as nn

def make_sequences(X, y, lookback):
    seqs, targets = [], []
    for i in range(lookback, len(X)):
        seqs.append(X[i - lookback:i])
        targets.append(y[i])
    return np.array(seqs), np.array(targets)

lookback = 24
print("lookback set to", lookback, "hours")

lookback set to 24 hours


In [27]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("using", device)

# take one fold and shape it into sequences
tr, te = folds[0]
X_all = dataset[feature_cols].values
y_all = dataset["y"].values

X_tr, y_tr = make_sequences(X_all[tr], y_all[tr], lookback)
X_te, y_te = make_sequences(X_all[te], y_all[te], lookback)

X_tr = torch.tensor(X_tr, dtype=torch.float32, device=device)
y_tr = torch.tensor(y_tr, dtype=torch.float32, device=device).unsqueeze(1)
X_te = torch.tensor(X_te, dtype=torch.float32, device=device)
y_te = torch.tensor(y_te, dtype=torch.float32, device=device).unsqueeze(1)

# define a small LSTM
class LSTMModel(nn.Module):
    def __init__(self, n_features, hidden=16):
        super().__init__()
        self.lstm = nn.LSTM(n_features, hidden, batch_first=True)
        self.head = nn.Linear(hidden, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        last = out[:, -1, :]
        return self.head(last)

model = LSTMModel(n_features=len(feature_cols)).to(device)

# train
loss_function = nn.MSELoss()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(20):
    model.train()
    opt.zero_grad()
    pred = model(X_tr)
    loss = loss_function(pred, y_tr)
    loss.backward()
    opt.step()
    if epoch % 5 == 0:
        print(f"epoch {epoch}: train loss {loss.item():.6f}")

# score
model.eval()
with torch.no_grad():
    pred_te = model(X_te).cpu().numpy().ravel()

truth = y_te.cpu().numpy().ravel()
acc = np.mean(np.sign(pred_te) == np.sign(truth))
ic  = np.corrcoef(pred_te, truth)[0, 1]
print(f"\nLSTM fold 0 -> acc: {acc:.4f}  ic: {ic:.4f}")
print(f"baseline bar -> acc: 0.5096  ic: 0.0104")

using cpu
epoch 0: train loss 0.034846
epoch 5: train loss 0.023771
epoch 10: train loss 0.014886
epoch 15: train loss 0.008190

LSTM fold 0 -> acc: 0.4732  ic: 0.0438
baseline bar -> acc: 0.5096  ic: 0.0104


In [28]:
def train_one_fold(tr, te, epochs=50, patience=5):
    X_tr, y_tr = make_sequences(X_all[tr], y_all[tr], lookback)
    X_te, y_te = make_sequences(X_all[te], y_all[te], lookback)

    # carve a small validation check off the END of training
    cut = int(len(X_tr) * 0.85)
    Xval, yval = X_tr[cut:], y_tr[cut:]
    X_tr, y_tr = X_tr[:cut], y_tr[:cut]

    #tensors
    to_t = lambda a: torch.tensor(a, dtype=torch.float32, device=device)
    X_tr, y_tr = to_t(X_tr), to_t(y_tr).unsqueeze(1)
    Xval, yval = to_t(Xval), to_t(yval).unsqueeze(1)
    X_te = to_t(X_te)

    model = LSTMModel(n_features=len(feature_cols)).to(device)
    loss_fn = nn.MSELoss()
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)

    best_val, best_state, wait = float("inf"), None, 0
    for epoch in range(epochs):
        model.train()
        opt.zero_grad()
        loss = loss_fn(model(X_tr), y_tr)
        loss.backward()
        opt.step()

        # check on the held-out slice
        model.eval()
        with torch.no_grad():
            vloss = loss_fn(model(Xval), yval).item()

        if vloss < best_val:
            best_val, best_state, wait = vloss, model.state_dict(), 0
        else:
            wait += 1
            if wait >= patience:
                break

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        pred = model(X_te).cpu().numpy().ravel()

    truth = y_te
    acc = np.mean(np.sign(pred) == np.sign(truth))
    ic  = np.corrcoef(pred, truth)[0, 1]
    return acc, ic


rows = []
for i, (tr, te) in enumerate(folds):
    acc, ic = train_one_fold(tr, te)
    rows.append({"fold": i, "acc_lstm": acc, "ic_lstm": ic})
    print(f"fold {i:2d}: acc {acc:.4f}  ic {ic:.4f}")

lstm_results = pd.DataFrame(rows)
print("\nLSTM means:")
print(lstm_results[["acc_lstm", "ic_lstm"]].mean().round(4))
print("baseline bar -> acc: 0.5096  ic: 0.0104")

fold  0: acc 0.5184  ic 0.0576
fold  1: acc 0.4816  ic -0.0153
fold  2: acc 0.4887  ic -0.0267
fold  3: acc 0.5113  ic 0.0018
fold  4: acc 0.4915  ic 0.0173
fold  5: acc 0.4972  ic -0.0110
fold  6: acc 0.5064  ic 0.0196
fold  7: acc 0.4958  ic 0.0111
fold  8: acc 0.5155  ic 0.0318
fold  9: acc 0.4823  ic 0.0062
fold 10: acc 0.5000  ic -0.0985
fold 11: acc 0.4774  ic -0.0344
fold 12: acc 0.5169  ic -0.0306
fold 13: acc 0.5056  ic 0.0033
fold 14: acc 0.5085  ic 0.0361
fold 15: acc 0.5028  ic -0.0220
fold 16: acc 0.5141  ic 0.0151
fold 17: acc 0.4972  ic 0.0052
fold 18: acc 0.5120  ic -0.0138
fold 19: acc 0.4936  ic 0.0340
fold 20: acc 0.5155  ic 0.0229
fold 21: acc 0.4718  ic -0.0053
fold 22: acc 0.4901  ic -0.0162
fold 23: acc 0.4965  ic -0.0159
fold 24: acc 0.4915  ic -0.0491
fold 25: acc 0.4767  ic -0.0284
fold 26: acc 0.5056  ic 0.0655
fold 27: acc 0.5028  ic -0.0122
fold 28: acc 0.4958  ic -0.0015
fold 29: acc 0.5056  ic 0.0141
fold 30: acc 0.4894  ic -0.0434
fold 31: acc 0.5000  ic

In [29]:
# Transformer
class TransformerModel(nn.Module):
    def __init__(self, n_features, d_model=16, nhead=2, dim_ff=32, lookback=24, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Linear(n_features, d_model)
        self.pos = nn.Parameter(torch.zeros(1, lookback, d_model))
        layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_ff,
            dropout=dropout, batch_first=True
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=1)
        self.head = nn.Linear(d_model, 1)

    def forward(self, x):
        x = self.input_proj(x) + self.pos
        out = self.encoder(x)
        last = out[:, -1, :]
        return self.head(last)

# Trainer
def train_fold(tr, te, make_model, epochs=50, patience=5):
    X_tr, y_tr = make_sequences(X_all[tr], y_all[tr], lookback)
    X_te, y_te = make_sequences(X_all[te], y_all[te], lookback)

    cut = int(len(X_tr) * 0.85)
    X_val, y_val = X_tr[cut:], y_tr[cut:]
    X_tr, y_tr = X_tr[:cut], y_tr[:cut]

    to_t = lambda a: torch.tensor(a, dtype=torch.float32, device=device)
    X_tr, y_tr = to_t(X_tr), to_t(y_tr).unsqueeze(1)
    X_val, y_val = to_t(X_val), to_t(y_val).unsqueeze(1)
    X_te = to_t(X_te)

    model = make_model().to(device)
    loss_fn = nn.MSELoss()
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)

    best_val, best_state, wait = float("inf"), None, 0
    for epoch in range(epochs):
        model.train()
        opt.zero_grad()
        loss = loss_fn(model(X_tr), y_tr)
        loss.backward()
        opt.step()

        model.eval()
        with torch.no_grad():
            vloss = loss_fn(model(X_val), y_val).item()
        if vloss < best_val:
            best_val, best_state, wait = vloss, model.state_dict(), 0
        else:
            wait += 1
            if wait >= patience:
                break

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        pred = model(X_te).cpu().numpy().ravel()

    acc = np.mean(np.sign(pred) == np.sign(y_te))
    ic = np.corrcoef(pred, y_te)[0, 1]
    return acc, ic

# Run
rows = []
for i, (tr, te) in enumerate(folds):
    acc, ic = train_fold(tr, te, make_model=lambda: TransformerModel(len(feature_cols), lookback=lookback))
    rows.append({"fold": i, "acc_tf": acc, "ic_tf": ic})
    print(f"fold {i:2d}: acc {acc:.4f}  ic {ic:.4f}")

tfResults = pd.DataFrame(rows)
print("\nTransformer means:")
print(tfResults[["acc_tf", "ic_tf"]].mean().round(4))
print("baseline bar -> acc: 0.5096  ic:  0.0104")
print("LSTM         -> acc: 0.4995  ic: -0.0040")

fold  0: acc 0.5056  ic -0.0303
fold  1: acc 0.5226  ic 0.0464
fold  2: acc 0.5184  ic 0.0371
fold  3: acc 0.5127  ic 0.0369
fold  4: acc 0.4965  ic 0.0156
fold  5: acc 0.4894  ic -0.0227
fold  6: acc 0.5085  ic -0.0202
fold  7: acc 0.5042  ic 0.0649
fold  8: acc 0.5035  ic 0.0182
fold  9: acc 0.5318  ic 0.0186
fold 10: acc 0.5198  ic 0.0055
fold 11: acc 0.4951  ic -0.0893
fold 12: acc 0.4972  ic 0.0016
fold 13: acc 0.5092  ic -0.0111
fold 14: acc 0.4993  ic -0.0244
fold 15: acc 0.5085  ic -0.0006
fold 16: acc 0.4993  ic 0.0267
fold 17: acc 0.4802  ic -0.0224
fold 18: acc 0.4979  ic 0.0010
fold 19: acc 0.4802  ic -0.0215
fold 20: acc 0.5000  ic 0.0219
fold 21: acc 0.5021  ic 0.0361
fold 22: acc 0.5120  ic 0.0199
fold 23: acc 0.4703  ic -0.0200
fold 24: acc 0.5162  ic 0.0147
fold 25: acc 0.4887  ic -0.0149
fold 26: acc 0.4915  ic -0.0253
fold 27: acc 0.4951  ic -0.0025
fold 28: acc 0.4894  ic -0.0038
fold 29: acc 0.5078  ic -0.0059
fold 30: acc 0.4929  ic -0.0005
fold 31: acc 0.5099  ic

I built a pipeline to test whether sequence models can predict crypto price direction. Using six and a half years of hourly Bitcoin data and proper purged walk-forward validation across 32 folds, I compared three models: a linear baseline (51.0% accuracy, 0.010 IC), an LSTM (49.9%, -0.004), and a transformer (49.9%, 0.001). The linear model won, both deep models landed at a coin flip, the expected result when predicting the noisiest target from just four features. Next I'm widening to about two dozen features to give the deep models real structure to find, then rerunning the same harness.

In [30]:
f = pd.DataFrame(index=data.index)

# Each hour own move
ret = np.log(data["close"]) - np.log(data["close"].shift(1))
f["ret_1"] = ret

# Momentum
for w in [3, 6, 12, 24, 48, 72, 168]:
    f[f"mom_{w}"] = ret.rolling(w).sum()

# Volatility
for w in [6, 24, 72, 168]:
    f[f"vol_{w}"] = ret.rolling(w).std()

# Volume
log_vol = np.log(data["volume"] + 1)
f["vol_chg"] = log_vol - log_vol.shift(1)
f["vol_rel"] = log_vol - log_vol.rolling(24).mean()

# Candle shape of the current bar
f["range"] = (data["high"] - data["low"]) / data["close"]
f["body"]  = (data["close"] - data["open"]) / data["open"]
f["uwick"] = (data["high"] - data[["open", "close"]].max(axis=1)) / data["close"]
f["lwick"] = (data[["open", "close"]].min(axis=1) - data["low"]) / data["close"]

# Price position
roll_low  = data["low"].rolling(24).min()
roll_high = data["high"].rolling(24).max()
f["pos_24"] = (data["close"] - roll_low) / (roll_high - roll_low + 1e-12)

# RSI
up, down = ret.clip(lower=0), (-ret).clip(lower=0)
avg_up, avg_down = up.rolling(14).mean(), down.rolling(14).mean()
f["rsi_14"] = avg_up / (avg_up + avg_down + 1e-12)

# Time
hour = data.index.hour
dow  = data.index.dayofweek
f["hour_sin"], f["hour_cos"] = np.sin(2*np.pi*hour/24), np.cos(2*np.pi*hour/24)
f["dow_sin"],  f["dow_cos"]  = np.sin(2*np.pi*dow/7),   np.cos(2*np.pi*dow/7)

f["y"] = data["y"]
print(f.shape)
f.tail()

(56200, 25)


,ret_1,mom_3,mom_6,mom_12,mom_24,mom_48,mom_72,mom_168,vol_6,vol_24,...,body,uwick,lwick,pos_24,rsi_14,hour_sin,hour_cos,dow_sin,dow_cos,y
time,,,,,,,,,,,,,,,,,,,,,
2026-05-31 18:00:00+00:00,-0.001436,-0.000700,-0.004288,-0.006111,-0.005956,0.003118,-0.000668,-0.040760,0.002129,0.001385,...,-0.001435,0.000053,0.000247,0.232613,0.290839,-1.000000,-1.836970e-16,-0.781831,0.62349,-0.000547
2026-05-31 19:00:00+00:00,-0.000547,0.000507,-0.004197,-0.004176,-0.006142,0.000587,0.001961,-0.042710,0.002130,0.001386,...,-0.000547,0.000629,0.001454,0.186625,0.309206,-0.965926,2.588190e-01,-0.781831,0.62349,0.002393
2026-05-31 20:00:00+00:00,0.002393,0.000410,0.001866,-0.002681,-0.003252,0.000896,0.002168,-0.038949,0.001860,0.001486,...,0.002396,0.000276,0.000000,0.387890,0.392645,-0.866025,5.000000e-01,-0.781831,0.62349,0.002071
2026-05-31 21:00:00+00:00,0.002071,0.003916,0.003217,0.000687,-0.001210,0.006561,0.001018,-0.030108,0.001996,0.001553,...,0.002073,0.000426,0.001529,0.562456,0.507403,-0.707107,7.071068e-01,-0.781831,0.62349,0.000705
2026-05-31 22:00:00+00:00,0.000705,0.005169,0.005675,0.000604,0.000830,0.006255,0.006510,-0.037940,0.001655,0.001535,...,0.000705,0.000716,0.002793,0.621954,0.502461,-0.500000,8.660254e-01,-0.781831,0.62349,-0.003661


In [31]:
feature_cols_big = [c for c in f.columns if c != "y"]

window = 24 * 30

normed_big = pd.DataFrame(index=f.index)
for col in feature_cols_big:
    roll = f[col].rolling(window)
    normed_big[col] = (f[col] - roll.mean()) / (roll.std() + 1e-8)

normed_big["y"] = f["y"]

# Final clean table
dataset_big = normed_big.dropna()

print(dataset_big.shape)
print(dataset_big.index.min(), "to", dataset_big.index.max())

(55313, 25)
2020-02-06 23:00:00+00:00 to 2026-05-31 22:00:00+00:00


In [32]:
feat = [c for c in dataset_big.columns if c != "y"]
dataset_big[feat] = dataset_big[feat].clip(-10, 10)
print("clipped to +/- 10; check ranges:")
print(dataset_big[feat].describe().loc[["min", "max"]].T.round(2))

clipped to +/- 10; check ranges:
            min    max
ret_1    -10.00  10.00
mom_3    -10.00  10.00
mom_6    -10.00   9.59
mom_12   -10.00   8.76
mom_24   -10.00   7.44
mom_48    -9.53   6.87
mom_72    -8.80   6.64
mom_168   -7.98   5.50
vol_6     -1.92  10.00
vol_24    -2.73  10.00
vol_72    -3.16  10.00
vol_168   -3.47  10.00
vol_chg  -10.00  10.00
vol_rel  -10.00   5.88
range     -1.79  10.00
body     -10.00  10.00
uwick     -1.30  10.00
lwick     -1.25  10.00
pos_24    -2.97   2.46
rsi_14    -3.19   3.15
hour_sin  -1.41   1.41
hour_cos  -1.42   1.41
dow_sin   -1.41   1.38
dow_cos   -1.33   1.43


/tmp/ipykernel_842/1435891976.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataset_big[feat] = dataset_big[feat].clip(-10, 10)


In [33]:
from sklearn.linear_model import Ridge

# Trainer
def train_fold(tr, te, X, y, make_model, epochs=50, patience=5):
    X_tr, y_tr = make_sequences(X[tr], y[tr], lookback)
    X_te, y_te = make_sequences(X[te], y[te], lookback)

    cut = int(len(X_tr) * 0.85)
    to_t = lambda a: torch.tensor(a, dtype=torch.float32, device=device)
    xt, ytt = to_t(X_tr[:cut]), to_t(y_tr[:cut]).unsqueeze(1)
    xv, yv  = to_t(X_tr[cut:]), to_t(y_tr[cut:]).unsqueeze(1)
    xe = to_t(X_te)

    m = make_model().to(device)
    opt = torch.optim.Adam(m.parameters(), lr=1e-3)
    lf = nn.MSELoss()

    best, state, wait = float("inf"), None, 0
    for _ in range(epochs):
        m.train(); opt.zero_grad()
        lf(m(xt), ytt).backward(); opt.step()
        m.eval()
        with torch.no_grad():
            v = lf(m(xv), yv).item()
        if v < best:
            best, state, wait = v, m.state_dict(), 0
        else:
            wait += 1
            if wait >= patience:
                break

    m.load_state_dict(state); m.eval()
    with torch.no_grad():
        p = m(xe).cpu().numpy().ravel()
    return np.mean(np.sign(p) == np.sign(y_te)), np.corrcoef(p, y_te)[0, 1]

# Run Big
feat_big = [c for c in dataset_big.columns if c != "y"]
Xb = dataset_big[feat_big].values
yb = dataset_big["y"].values
n_feat_big = len(feat_big)

folds_big = list(walk_forward_splits(len(dataset_big), train_size, test_size, purge, embargo))
print("folds:", len(folds_big), " features:", n_feat_big)

rows = []
for i, (tr, te) in enumerate(folds_big):
    ridge = Ridge(alpha=1.0).fit(Xb[tr], yb[tr])
    p = ridge.predict(Xb[te])
    acc_r = np.mean(np.sign(p) == np.sign(yb[te]))
    ic_r  = np.corrcoef(p, yb[te])[0, 1]

    acc_l, ic_l = train_fold(tr, te, Xb, yb, lambda: LSTMModel(n_feat_big))
    acc_t, ic_t = train_fold(tr, te, Xb, yb, lambda: TransformerModel(n_feat_big, lookback=lookback))

    rows.append({"fold": i,
                 "acc_ridge": acc_r, "ic_ridge": ic_r,
                 "acc_lstm": acc_l, "ic_lstm": ic_l,
                 "acc_tf": acc_t, "ic_tf": ic_t})
    print(f"fold {i:2d}: ridge {acc_r:.3f}/{ic_r:+.3f}   lstm {acc_l:.3f}/{ic_l:+.3f}   tf {acc_t:.3f}/{ic_t:+.3f}")

big = pd.DataFrame(rows)
print("\nmeans (acc / ic):")
print(f"ridge:       {big.acc_ridge.mean():.4f}  {big.ic_ridge.mean():+.4f}")
print(f"lstm:        {big.acc_lstm.mean():.4f}  {big.ic_lstm.mean():+.4f}")
print(f"transformer: {big.acc_tf.mean():.4f}  {big.ic_tf.mean():+.4f}")

folds: 32  features: 24
fold  0: ridge 0.502/+0.068   lstm 0.501/+0.008   tf 0.482/-0.024
fold  1: ridge 0.533/+0.081   lstm 0.493/-0.017   tf 0.511/+0.039
fold  2: ridge 0.545/+0.059   lstm 0.497/+0.016   tf 0.516/-0.002
fold  3: ridge 0.515/-0.022   lstm 0.505/+0.014   tf 0.499/-0.040
fold  4: ridge 0.503/-0.032   lstm 0.508/-0.003   tf 0.504/-0.023
fold  5: ridge 0.485/-0.025   lstm 0.492/+0.018   tf 0.513/-0.022
fold  6: ridge 0.517/+0.046   lstm 0.508/-0.046   tf 0.517/-0.010
fold  7: ridge 0.501/+0.035   lstm 0.521/+0.031   tf 0.481/-0.037
fold  8: ridge 0.490/-0.021   lstm 0.501/+0.019   tf 0.509/+0.013
fold  9: ridge 0.524/+0.059   lstm 0.492/-0.018   tf 0.498/-0.007
fold 10: ridge 0.508/-0.083   lstm 0.511/-0.010   tf 0.505/+0.042
fold 11: ridge 0.494/-0.021   lstm 0.472/+0.011   tf 0.504/+0.012
fold 12: ridge 0.502/+0.019   lstm 0.518/+0.012   tf 0.514/+0.023
fold 13: ridge 0.494/-0.006   lstm 0.507/+0.018   tf 0.519/+0.022
fold 14: ridge 0.483/-0.026   lstm 0.505/+0.005   tf

I widened the feature set from four to two dozen, spanning momentum, volatility, volume, candle shape, price position, an RSI style oscillator, and time, then reran all three models. The linear baseline improved to 51.4% accuracy with a 0.024 IC, while the LSTM and transformer stayed at a coin flip near 49.8%. More features helped the simple model and did nothing for the deep ones, since the added features are all derived from the same price columns and carry no signal the deep models couldn't already find. I am switching from hourly to daily Bitcoin bars and adding free daily on-chain features (like exchange flows and active addresses) to test whether new data, not new models, can finally move direction prediction above the coin-flip baseline.

In [34]:
url = "https://community-api.coinmetrics.io/v4/timeseries/asset-metrics"
params = {
    "assets": "btc",
    "metrics": "AdrActCnt,TxCnt,HashRate,SplyCur",
    "frequency": "1d",
    "start_time": "2020-01-01",
    "page_size": 10000,
}

r = requests.get(url, params=params)
print(r.status_code)
onchain = pd.DataFrame(r.json()["data"])
print(onchain.shape)
onchain.head()

200
(2373, 6)


,asset,time,AdrActCnt,HashRate,SplyCur,TxCnt
0,btc,2020-01-01T00:00:00.000000000Z,524360,112929774.50430365,18135692.32139018,251867
1,btc,2020-01-02T00:00:00.000000000Z,671016,96717718.31238091,18137454.82135472,291935
2,btc,2020-01-03T00:00:00.000000000Z,721747,115924073.72193182,18139567.32134926,323153
3,btc,2020-01-04T00:00:00.000000000Z,611397,115238132.457305,18141667.32134926,282301
4,btc,2020-01-05T00:00:00.000000000Z,597061,111808426.13417092,18143704.82134926,288212


In [35]:
onchain = pd.DataFrame(r.json()["data"])
onchain["time"] = pd.to_datetime(onchain["time"], utc=True).dt.floor("D")
onchain = onchain.set_index("time").drop(columns=["asset"]).astype(float)

print(onchain.dtypes)
onchain.head()

AdrActCnt    float64
HashRate     float64
SplyCur      float64
TxCnt        float64
dtype: object


,AdrActCnt,HashRate,SplyCur,TxCnt
time,,,,
2020-01-01 00:00:00+00:00,524360.0,1.129298e+08,1.813569e+07,251867.0
2020-01-02 00:00:00+00:00,671016.0,9.671772e+07,1.813745e+07,291935.0
2020-01-03 00:00:00+00:00,721747.0,1.159241e+08,1.813957e+07,323153.0
2020-01-04 00:00:00+00:00,611397.0,1.152381e+08,1.814167e+07,282301.0
2020-01-05 00:00:00+00:00,597061.0,1.118084e+08,1.814370e+07,288212.0


In [36]:
daily = pd.DataFrame()
daily["open"]   = data["open"].resample("1D").first()
daily["high"]   = data["high"].resample("1D").max()
daily["low"]    = data["low"].resample("1D").min()
daily["close"]  = data["close"].resample("1D").last()
daily["volume"] = data["volume"].resample("1D").sum()
daily = daily.dropna()
print(daily.shape)
daily.tail()

(2343, 5)


,open,high,low,close,volume
time,,,,,
2026-05-27 00:00:00+00:00,75930.01,76174.15,74243.99,74449.30,16877.77483
2026-05-28 00:00:00+00:00,74449.31,74590.77,72582.82,73617.51,21274.02026
2026-05-29 00:00:00+00:00,73617.52,74514.10,72512.49,73460.78,18686.74141
2026-05-30 00:00:00+00:00,73460.78,74143.76,73216.00,73884.38,7515.40909
2026-05-31 00:00:00+00:00,73884.38,74275.66,73400.00,73944.62,6611.14779


In [37]:
df_d = daily.join(onchain, how="inner")
print(df_d.shape)
print(df_d.columns.tolist())
df_d.tail()

(2343, 9)
['open', 'high', 'low', 'close', 'volume', 'AdrActCnt', 'HashRate', 'SplyCur', 'TxCnt']


,open,high,low,close,volume,AdrActCnt,HashRate,SplyCur,TxCnt
time,,,,,,,,,
2026-05-27 00:00:00+00:00,75930.01,76174.15,74243.99,74449.30,16877.77483,608435.0,9.507084e+08,2.003518e+07,646713.0
2026-05-28 00:00:00+00:00,74449.31,74590.77,72582.82,73617.51,21274.02026,622022.0,1.100105e+09,2.003569e+07,783577.0
2026-05-29 00:00:00+00:00,73617.52,74514.10,72512.49,73460.78,18686.74141,669839.0,9.601639e+08,2.003612e+07,634214.0
2026-05-30 00:00:00+00:00,73460.78,74143.76,73216.00,73884.38,7515.40909,573532.0,9.808662e+08,2.003657e+07,657986.0
2026-05-31 00:00:00+00:00,73884.38,74275.66,73400.00,73944.62,6611.14779,535317.0,9.946812e+08,2.003702e+07,715768.0


In [38]:
fd = pd.DataFrame(index=df_d.index)

dret = np.log(df_d["close"]) - np.log(df_d["close"].shift(1))

# Price features
fd["ret_1"] = dret
for w in [2, 3, 5, 10, 20]:
    fd[f"mom_{w}"] = dret.rolling(w).sum()
for w in [5, 10, 20]:
    fd[f"vol_{w}"] = dret.rolling(w).std()

# On-chain features
for col in ["AdrActCnt", "TxCnt", "HashRate"]:
    fd[f"{col}_chg"] = np.log(df_d[col]) - np.log(df_d[col].shift(1))
    fd[f"{col}_rel"] = np.log(df_d[col]) - np.log(df_d[col]).rolling(30).mean()

# Target: next day return
fd["y"] = dret.shift(-1)
fd = fd.iloc[:-1]

print(fd.shape)
fd.tail()

(2342, 16)


,ret_1,mom_2,mom_3,mom_5,mom_10,mom_20,vol_5,vol_10,vol_20,AdrActCnt_chg,AdrActCnt_rel,TxCnt_chg,TxCnt_rel,HashRate_chg,HashRate_rel,y
time,,,,,,,,,,,,,,,,
2026-05-26 00:00:00+00:00,-0.018167,-0.014837,-0.010768,-0.021955,-0.028793,-0.070141,0.017677,0.012775,0.014065,0.026749,-0.023339,-0.220817,0.028741,-0.171472,-0.031353,-0.019694
2026-05-27 00:00:00+00:00,-0.019694,-0.037860,-0.034530,-0.014537,-0.039613,-0.071983,0.015466,0.013760,0.014169,-0.027632,-0.049324,0.031559,0.056548,0.007168,-0.024912,-0.011235
2026-05-28 00:00:00+00:00,-0.011235,-0.030929,-0.049096,-0.041697,-0.044947,-0.085555,0.011446,0.013945,0.014195,0.022085,-0.027095,0.191967,0.238175,0.145954,0.115690,-0.002131
2026-05-29 00:00:00+00:00,-0.002131,-0.013367,-0.033060,-0.047897,-0.044900,-0.093719,0.010011,0.013946,0.013999,0.074062,0.044979,-0.211483,0.009085,-0.136057,-0.024194,0.005750
2026-05-30 00:00:00+00:00,0.005750,0.003619,-0.007617,-0.045477,-0.048450,-0.106776,0.010817,0.013597,0.013123,-0.155224,-0.104307,0.036797,0.030411,0.021332,-0.004151,0.000815


In [39]:
from sklearn.linear_model import Ridge

feat_d = [c for c in fd.columns if c != "y"]
nd = pd.DataFrame(index=fd.index)
for col in feat_d:
    roll = fd[col].rolling(90)
    nd[col] = (fd[col] - roll.mean()) / (roll.std() + 1e-8)
nd["y"] = fd["y"]
dataset_d = nd.dropna()
dataset_d[feat_d] = dataset_d[feat_d].clip(-10, 10)

X_d = dataset_d[feat_d].values
y_d = dataset_d["y"].values
n_feat_d = len(feat_d)

lookback = 10
folds_d = list(walk_forward_splits(len(dataset_d), 365, 90, 5, 5))
print("folds:", len(folds_d), " features:", n_feat_d)

rows = []
for i, (tr, te) in enumerate(folds_d):
    ridge = Ridge(alpha=1.0).fit(X_d[tr], y_d[tr])
    p = ridge.predict(X_d[te])
    acc_r = np.mean(np.sign(p) == np.sign(y_d[te]))
    ic_r  = np.corrcoef(p, y_d[te])[0, 1]

    acc_l, ic_l = train_fold(tr, te, X_d, y_d, lambda: LSTMModel(n_feat_d))
    acc_t, ic_t = train_fold(tr, te, X_d, y_d, lambda: TransformerModel(n_feat_d, lookback=lookback))

    rows.append({"acc_ridge": acc_r, "ic_ridge": ic_r,
                 "acc_lstm": acc_l, "ic_lstm": ic_l, "acc_tf": acc_t, "ic_tf": ic_t})

dd = pd.DataFrame(rows)
print(f"ridge:       {dd.acc_ridge.mean():.4f}  {dd.ic_ridge.mean():+.4f}")
print(f"lstm:        {dd.acc_lstm.mean():.4f}  {dd.ic_lstm.mean():+.4f}")
print(f"transformer: {dd.acc_tf.mean():.4f}  {dd.ic_tf.mean():+.4f}")

folds: 20  features: 15


/tmp/ipykernel_842/1644680282.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataset_d[feat_d] = dataset_d[feat_d].clip(-10, 10)


ridge:       0.4939  -0.0440
lstm:        0.5200  +0.0238
transformer: 0.5144  -0.0401


On the daily run with price plus basic on-chain metrics, all three models came in below a coin flip with negative ICs, ridge 0.494/-0.044, LSTM 0.484/-0.030, transformer 0.494/-0.048. The basic free on-chain features (active addresses, transactions, hash rate) did not improve direction and arguably hurt, though with only 20 folds on thin daily data the means are noisy. Concluding the limit was the quality of information, I'm now pulling actual exchange flow data from Dune, inflows and outflows of Bitcoin to exchanges, which is the positioning signal practitioners rate for direction, to test whether it moves the result where network-health metrics couldn't.

In [40]:
!pip -q install dune-client

from dune_client.client import DuneClient
from getpass import getpass

dune = DuneClient(getpass("paste Dune key: "))
res = dune.get_latest_result(7822977)
flows = pd.DataFrame(res.result.rows)

flows["hour"] = pd.to_datetime(flows["hour"], utc=True)
flows = flows.sort_values("hour").set_index("hour")
print(flows.shape)
print(flows.index.min(), "to", flows.index.max())
flows.head()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 kB 5.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
paste Dune key: ··········
(56728, 4)
2020-01-01 00:00:00+00:00 to 2026-06-28 18:00:00+00:00


,inflow_btc,net_flow_btc,outflow_btc,symbol
hour,,,,
2020-01-01 00:00:00+00:00,0.0,-0.000044,0.000044,BTC
2020-01-01 01:00:00+00:00,0.0,-0.000054,0.000054,BTC
2020-01-01 02:00:00+00:00,0.0,-0.000048,0.000048,BTC
2020-01-01 03:00:00+00:00,0.0,-0.000019,0.000019,BTC
2020-01-01 04:00:00+00:00,0.0,-0.000009,0.000009,BTC


In [41]:
fl = pd.DataFrame()
fl["inflow"]  = flows["inflow_btc"].resample("1D").sum()
fl["outflow"] = flows["outflow_btc"].resample("1D").sum()
fl["netflow"] = flows["net_flow_btc"].resample("1D").sum()
fl = fl.dropna()
print(fl.shape, fl.index.min(), "to", fl.index.max())
fl.tail()

(2371, 3) 2020-01-01 00:00:00+00:00 to 2026-06-28 00:00:00+00:00


,inflow,outflow,netflow
hour,,,
2026-06-24 00:00:00+00:00,0.000000,0.001299,-0.001299
2026-06-25 00:00:00+00:00,0.000000,0.001679,-0.001679
2026-06-26 00:00:00+00:00,0.000000,0.001277,-0.001277
2026-06-27 00:00:00+00:00,0.000268,0.001337,-0.001070
2026-06-28 00:00:00+00:00,0.001087,0.001096,-0.000008


In [42]:
# flow features
ff = pd.DataFrame(index=fl.index)
ff["netflow_rel"] = fl["netflow"] - fl["netflow"].rolling(30).mean()      # vs last month
ff["netflow_z"]   = (fl["netflow"] - fl["netflow"].rolling(30).mean()) / (fl["netflow"].rolling(30).std() + 1e-9)
ff["outflow_chg"] = np.log(fl["outflow"] + 1e-9) - np.log(fl["outflow"].shift(1) + 1e-9)
ff["flow_5"]      = fl["netflow"].rolling(5).mean()                        # 5-day flow trend


df_dl = df_d.join(ff, how="inner").dropna()
print(df_dl.shape)
print(df_dl.columns.tolist())
df_dl.tail()

(2314, 13)
['open', 'high', 'low', 'close', 'volume', 'AdrActCnt', 'HashRate', 'SplyCur', 'TxCnt', 'netflow_rel', 'netflow_z', 'outflow_chg', 'flow_5']


,open,high,low,close,volume,AdrActCnt,HashRate,SplyCur,TxCnt,netflow_rel,netflow_z,outflow_chg,flow_5
2026-05-27 00:00:00+00:00,75930.01,76174.15,74243.99,74449.30,16877.77483,608435.0,9.507084e+08,2.003518e+07,646713.0,-0.000135,-0.563334,-0.018526,-0.000562
2026-05-28 00:00:00+00:00,74449.31,74590.77,72582.82,73617.51,21274.02026,622022.0,1.100105e+09,2.003569e+07,783577.0,-0.000218,-0.913142,0.108455,-0.000626
2026-05-29 00:00:00+00:00,73617.52,74514.10,72512.49,73460.78,18686.74141,669839.0,9.601639e+08,2.003612e+07,634214.0,-0.000082,-0.357204,-0.202513,-0.000685
2026-05-30 00:00:00+00:00,73460.78,74143.76,73216.00,73884.38,7515.40909,573532.0,9.808662e+08,2.003657e+07,657986.0,0.000119,0.593855,-0.427284,-0.000660
2026-05-31 00:00:00+00:00,73884.38,74275.66,73400.00,73944.62,6611.14779,535317.0,9.946812e+08,2.003702e+07,715768.0,0.000125,0.705120,-0.067764,-0.000594


In [43]:
from sklearn.linear_model import Ridge

fa = pd.DataFrame(index=df_dl.index)
dret = np.log(df_dl["close"]) - np.log(df_dl["close"].shift(1))
fa["ret_1"] = dret
for w in [2, 3, 5, 10, 20]:
    fa[f"mom_{w}"] = dret.rolling(w).sum()
for w in [5, 10, 20]:
    fa[f"vol_{w}"] = dret.rolling(w).std()
for col in ["AdrActCnt", "TxCnt", "HashRate"]:
    fa[f"{col}_chg"] = np.log(df_dl[col]) - np.log(df_dl[col].shift(1))
    fa[f"{col}_rel"] = np.log(df_dl[col]) - np.log(df_dl[col]).rolling(30).mean()
for col in ["netflow_rel", "netflow_z", "outflow_chg", "flow_5"]:
    fa[col] = df_dl[col]
fa["y"] = dret.shift(-1)
fa = fa.iloc[:-1]

# normalize
feat_a = [c for c in fa.columns if c != "y"]
na = pd.DataFrame(index=fa.index)
for col in feat_a:
    roll = fa[col].rolling(90)
    na[col] = (fa[col] - roll.mean()) / (roll.std() + 1e-8)
na["y"] = fa["y"]
dataset_a = na.dropna()
dataset_a[feat_a] = dataset_a[feat_a].clip(-10, 10)

# run all three
X_a = dataset_a[feat_a].values
y_a = dataset_a["y"].values
n_feat_a = len(feat_a)
lookback = 10
folds_a = list(walk_forward_splits(len(dataset_a), 365, 90, 5, 5))
print("folds:", len(folds_a), " features:", n_feat_a)

rows = []
for tr, te in folds_a:
    ridge = Ridge(alpha=1.0).fit(X_a[tr], y_a[tr])
    p = ridge.predict(X_a[te])
    acc_r = np.mean(np.sign(p) == np.sign(y_a[te]))
    ic_r  = np.corrcoef(p, y_a[te])[0, 1]
    acc_l, ic_l = train_fold(tr, te, X_a, y_a, lambda: LSTMModel(n_feat_a))
    acc_t, ic_t = train_fold(tr, te, X_a, y_a, lambda: TransformerModel(n_feat_a, lookback=lookback))
    rows.append({"acc_ridge": acc_r, "ic_ridge": ic_r, "acc_lstm": acc_l,
                 "ic_lstm": ic_l, "acc_tf": acc_t, "ic_tf": ic_t})

da = pd.DataFrame(rows)
print(f"ridge:       {da.acc_ridge.mean():.4f}  {da.ic_ridge.mean():+.4f}")
print(f"lstm:        {da.acc_lstm.mean():.4f}  {da.ic_lstm.mean():+.4f}")
print(f"transformer: {da.acc_tf.mean():.4f}  {da.ic_tf.mean():+.4f}")

folds: 20  features: 19


/tmp/ipykernel_842/837280437.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataset_a[feat_a] = dataset_a[feat_a].clip(-10, 10)


ridge:       0.5000  -0.0355
lstm:        0.5012  +0.0167
transformer: 0.4750  +0.0024
